In [ ]:
import pandas as pd
import numpy as np
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '..'))
from utils import get_spike_activity, sdf, sdf_mean
import json
import matplotlib.pyplot as plt
import pickle
import seaborn as sns
import matplotlib.patches as patches
import scipy.stats as st
import h5py

n_sim = input("Number of simulations: ")
folder_path = "results"  # Set the folder name
thr_perc = 0.30

with open("/g100_work/EIRI_E_POLIM2/no_paper/NODS/network_configuration.json", "r") as json_file:
    net_config = json.load(json_file)
CS_burst_dur = net_config["devices"]["CS"]["parameters"]["burst_dur"]
CS_start_first = float(net_config["devices"]["CS"]["parameters"]["start_first"])
between_start = net_config["devices"]["CS"]["parameters"]["between_start"]
n_trials = net_config["devices"]["CS"]["parameters"]["n_trials"]
US_start_first = float(net_config["devices"]["US"]["parameters"]["start_first"])
CS_color = net_config["colors"]["CS"]
US_color = net_config["colors"]["US"]
cell_color = net_config["cell_types"]["purkinje_cell"]["color"][0]
with_NO_color = net_config["devices"]["nNOS"]["color"][0]

palette = list(reversed(sns.color_palette("viridis", n_trials).as_hex()))
sm = plt.cm.ScalarMappable(cmap="viridis_r", norm=plt.Normalize(vmin=0, vmax=n_trials))

In [ ]:
file_path = '/g100_work/EIRI_E_POLIM2/no_paper/NODS/data/pfs-PC_CS_40.pkl'
with open(file_path, 'rb') as file:
    id_granule = pickle.load(file)
    file.close()

rel_dist_path = '/g100_work/EIRI_E_POLIM2/no_paper/NODS/data/relative_dist.csv'
# source_id, nos_id, ev_points_id, d, cluster
relative_dist = pd.read_csv(rel_dist_path, header = None)
relative_dist = relative_dist.iloc[:,1:]
relative_dist.columns = ['source_id', 'nos_id', 'ev_points_id', 'd', 'cluster']
matched_grc = relative_dist[relative_dist['source_id'].isin(id_granule)]
  
thr_perc = 0.30
pc_ids = np.unique(relative_dist['cluster'])
total_synapses_per_pc = relative_dist.groupby('cluster').size()
matched_synapses_per_pc = matched_grc.groupby('cluster').size()
pc_ratio = (matched_synapses_per_pc / total_synapses_per_pc).fillna(0)
pc_ratio_df = pc_ratio.reset_index()
pc_ratio_df.columns = ['PC_cluster', 'matched_ratio']
pc_ratio_df['CS_syn'] = pc_ratio_df['matched_ratio'].apply(
    lambda x: 'CS' if x >= thr_perc else 'no_CS'
)

cs_pc_ids = pc_ratio_df[pc_ratio_df['CS_syn'] == 'CS']['PC_cluster'].values
no_cs_pc_ids = pc_ratio_df[pc_ratio_df['CS_syn'] == 'no_CS']['PC_cluster'].values

In [ ]:
result_path = "/g100_scratch/userexternal/csartor1/results/Paper/"
cell = "pc"
folder_path = result_path + f"without_NO/"
path_m_p = 'minus100_plus10'
folder_path_NO = result_path + f"{NO_folder}/{path_m_p}/"

os.makedirs(folder_save_path+f'_{path_m_p}', exist_ok = True)
step = 5

In [ ]:
m_ratio_sim_cs, m_ratio_sim_no_cs, m_ratio_sim_cs_NO, m_ratio_sim_no_cs_NO = [], [], [], []

for noise in [0,4,8]:

    file_save_path = folder_save_path+f'saved_files_{noise}Hz.hdf5'
    if os.path.isfile(file_save_path):
        with h5py.File(file_save_path, 'r') as f:
            median_sdf_cs = f['pc_sdf/median_sdf_cs'][:]
            median_sdf_no_cs = f['pc_sdf/median_sdf_no_cs'][:]
            median_sdf_cs_NO = f['pc_sdf/median_sdf_cs_NO'][:]
            median_sdf_no_cs_NO = f['pc_sdf/median_sdf_no_cs_NO'][:]

            sdf_change_bs_cs = f['sdf_change/sdf_change_bs_cs'][:]
            sdf_change_bs_no_cs = f['sdf_change/sdf_change_bs_no_cs'][:]
            sdf_change_bs_cs_NO = f['sdf_change/sdf_change_bs_cs_NO'][:]
            sdf_change_bs_no_cs_NO = f['sdf_change/sdf_change_bs_no_cs_NO'][:]
            sdf_change_cr_cs = f['sdf_change/sdf_change_cr_cs'][:]
            sdf_change_cr_no_cs = f['sdf_change/sdf_change_cr_no_cs'][:]
            sdf_change_cr_cs_NO = f['sdf_change/sdf_change_cr_cs_NO'][:]
            sdf_change_cr_no_cs_NO = f['sdf_change/sdf_change_cr_no_cs_NO'][:]

            m_ratio_sim_cs = f['m_ratio/m_ratio_sim_cs']
            m_ratio_sim_no_cs = f['m_ratio/m_ratio_sim_no_cs']
            m_ratio_sim_cs_NO = f['m_ratio/m_ratio_sim_cs_NO']
            m_ratio_sim_no_cs_NO = f['m_ratio/m_ratio_sim_no_cs_NO']

    else:
        # PC Processing
        sdf_mean_trials_simulations_cs = []
        sdf_mean_trials_simulations_no_cs = []
        
        sdf_mean_trials_simulations_cs_NO = []
        sdf_mean_trials_simulations_no_cs_NO = []
    
        # sdf change
    
        sdf_change_bs_cs = []
        sdf_change_cr_cs = []
    
        sdf_change_bs_no_cs = []
        sdf_change_cr_no_cs = []
    
        sdf_change_bs_cs_NO = []
        sdf_change_cr_cs_NO = []
    
        sdf_change_bs_no_cs_NO = []
        sdf_change_cr_no_cs_NO = []
    
        m_ratio_cs, m_ratio_no_cs, m_ratio_cs_NO, m_ratio_no_cs_NO = [], [], [], []
        
        
        for k in range(n_sim):
            # Process PC data
            file_path = folder_path + f'simulation_{noise}Hz_sim{k}'
            spk = get_spike_activity(cell_name=cell, path=file_path)
            sdf_mean_cs = []
            sdf_mean_no_cs = []
            spk_cs = spk[np.isin(spk[:, 0], cs_pc_ids)]
            spk_no_cs = spk[np.isin(spk[:, 0], no_cs_pc_ids)]
    
            file_path_NO = folder_path_NO + f'{noise}Hz/sim{k}'
            spk_NO = get_spike_activity(cell_name=cell, path=file_path_NO)
            sdf_mean_cs_NO = []
            sdf_mean_no_cs_NO = []
            spk_cs_NO = spk_NO[np.isin(spk_NO[:, 0], cs_pc_ids)]
            spk_no_cs_NO = spk_NO[np.isin(spk_NO[:, 0], no_cs_pc_ids)]
            
            for trial in range(n_trials):
                start = trial * between_start
                stop = CS_start_first + CS_burst_dur + trial * between_start
            
                # PC - CS PCs
                sdf_cs = sdf(start=start, stop=stop, spk=spk_cs, step=step)
                sdf_mean_cs.append(sdf_mean(sdf_cs))
            
                # PC - no_CS PCs
                sdf_no_cs = sdf(start=start, stop=stop, spk=spk_no_cs, step=step)
                sdf_mean_no_cs.append(sdf_mean(sdf_no_cs))
    
                # PC - CS PCs NO
                sdf_cs_NO = sdf(start=start, stop=stop, spk=spk_cs_NO, step=step)
                sdf_mean_cs_NO.append(sdf_mean(sdf_cs_NO))
            
                # PC - no_CS PCs NO
                sdf_no_cs_NO = sdf(start=start, stop=stop, spk=spk_no_cs_NO, step=step)
                sdf_mean_no_cs_NO.append(sdf_mean(sdf_no_cs_NO))
                
            # PC data
            sdf_mean_over_trials_cs = np.array(sdf_mean_cs)
            sdf_mean_over_trials_no_cs = np.array(sdf_mean_no_cs)
            
            sdf_mean_trials_simulations_cs.append(sdf_mean_over_trials_cs)
            sdf_mean_trials_simulations_no_cs.append(sdf_mean_over_trials_no_cs)
    
            sdf_mean_over_trials_cs_NO = np.array(sdf_mean_cs_NO)
            sdf_mean_over_trials_no_cs_NO = np.array(sdf_mean_no_cs_NO)
            
            sdf_mean_trials_simulations_cs_NO.append(sdf_mean_over_trials_cs_NO)
            sdf_mean_trials_simulations_no_cs_NO.append(sdf_mean_over_trials_no_cs_NO)
    
            # sdf change
            bs_cs = np.mean(sdf_mean_over_trials_cs[:,150:200],axis = 1)
            cr_cs = np.mean(sdf_mean_over_trials_cs[:,250:300],axis = 1)
            sim_bs_cs = bs_cs[1:] - bs_cs[1]
            sim_cr_cs = cr_cs[1:] - cr_cs[1]
            sdf_change_bs_cs.append(sim_bs_cs)
            sdf_change_cr_cs.append(sim_cr_cs)
    
            bs_no_cs = np.mean(sdf_mean_over_trials_no_cs[:,150:200],axis = 1)
            cr_no_cs = np.mean(sdf_mean_over_trials_no_cs[:,250:300],axis = 1)
            sim_bs_no_cs = bs_no_cs[1:] - bs_no_cs[1]
            sim_cr_no_cs = cr_no_cs[1:] - cr_no_cs[1]
            sdf_change_bs_no_cs.append(sim_bs_no_cs)
            sdf_change_cr_no_cs.append(sim_cr_no_cs)
    
            bs_cs_NO = np.mean(sdf_mean_over_trials_cs_NO[:,150:200],axis = 1)
            cr_cs_NO = np.mean(sdf_mean_over_trials_cs_NO[:,250:300],axis = 1)
            sim_bs_cs_NO = bs_cs_NO[1:] - bs_cs_NO[1]
            sim_cr_cs_NO = cr_cs_NO[1:] - cr_cs_NO[1]
            sdf_change_bs_cs_NO.append(sim_bs_cs_NO)
            sdf_change_cr_cs_NO.append(sim_cr_cs_NO)
            
            bs_no_cs_NO = np.mean(sdf_mean_over_trials_no_cs_NO[:,150:200],axis = 1)
            cr_no_cs_NO = np.mean(sdf_mean_over_trials_no_cs_NO[:,250:300],axis = 1)
            sim_bs_no_cs_NO = bs_no_cs_NO[1:] - bs_no_cs_NO[1]
            sim_cr_no_cs_NO = cr_no_cs_NO[1:] - cr_no_cs_NO[1]
            sdf_change_bs_no_cs_NO.append(sim_bs_no_cs_NO)
            sdf_change_cr_no_cs_NO.append(sim_cr_no_cs_NO)
    
            # m_cr/m_bs
            trials = np.arange(0,n_trials-1)
            m_bs_cs, q_bs_cs = np.polyfit(trials, sim_bs_cs, 1)
            m_cr_cs, q_cr_cs = np.polyfit(trials, sim_cr_cs, 1)
            m_bs_no_cs, q_bs_no_cs = np.polyfit(trials, sim_bs_no_cs, 1)
            m_cr_no_cs, q_cr_no_cs = np.polyfit(trials, sim_cr_no_cs, 1)
            m_bs_cs_NO, q_bs_cs_NO = np.polyfit(trials, sim_bs_cs_NO, 1)
            m_cr_cs_NO, q_cr_cs_NO = np.polyfit(trials, sim_cr_cs_NO, 1)
            m_bs_no_cs_NO, q_bs_no_cs_NO = np.polyfit(trials, sim_bs_no_cs_NO, 1)
            m_cr_no_cs_NO, q_cr_no_cs_NO = np.polyfit(trials, sim_cr_no_cs_NO, 1)
    
            m_ratio_cs.append(m_cr_cs/m_bs_cs)
            m_ratio_no_cs.append(m_cr_no_cs/m_bs_no_cs)
            m_ratio_cs_NO.append(m_cr_cs_NO/m_bs_cs_NO)
            m_ratio_no_cs_NO.append(m_cr_no_cs_NO/m_bs_no_cs_NO)
            
        # PC data processing
        stack_sdf_mean_cs = np.stack(sdf_mean_trials_simulations_cs, axis=0)
        stack_sdf_mean_no_cs = np.stack(sdf_mean_trials_simulations_no_cs, axis=0)
        
        stack_sdf_mean_cs_NO = np.stack(sdf_mean_trials_simulations_cs_NO, axis=0)
        stack_sdf_mean_no_cs_NO = np.stack(sdf_mean_trials_simulations_no_cs_NO, axis=0)
        
        median_sdf_cs = np.median(stack_sdf_mean_cs, axis=0)
        median_sdf_no_cs = np.median(stack_sdf_mean_no_cs, axis=0)
    
        median_sdf_cs_NO = np.median(stack_sdf_mean_cs_NO, axis=0)
        median_sdf_no_cs_NO = np.median(stack_sdf_mean_no_cs_NO, axis=0)

        # ratio m_cr/m_bs
        m_ratio_sim_cs.append(np.stack(m_ratio_cs))
        m_ratio_sim_cs_NO.append(np.stack(m_ratio_cs_NO))
    
        m_ratio_sim_no_cs.append(np.stack(m_ratio_no_cs))
        m_ratio_sim_no_cs_NO.append(np.stack(m_ratio_no_cs_NO))
    
        # sdf change processing
    
        sdf_change_bs_cs = np.stack(sdf_change_bs_cs, axis=0)
        sdf_change_cr_cs = np.stack(sdf_change_cr_cs, axis=0)
        
        sdf_change_bs_cs_NO = np.stack(sdf_change_bs_cs_NO, axis=0)
        sdf_change_cr_cs_NO = np.stack(sdf_change_cr_cs_NO, axis=0)
    
        sdf_change_bs_no_cs = np.stack(sdf_change_bs_no_cs, axis=0)
        sdf_change_cr_no_cs = np.stack(sdf_change_cr_no_cs, axis=0)
        
        sdf_change_bs_no_cs_NO = np.stack(sdf_change_bs_no_cs_NO, axis=0)
        sdf_change_cr_no_cs_NO = np.stack(sdf_change_cr_no_cs_NO, axis=0)
        
    median_bs_cs = np.median(sdf_change_bs_cs, axis=0)
    median_cr_cs = np.median(sdf_change_cr_cs, axis=0)
    
    median_bs_cs_NO = np.median(sdf_change_bs_cs_NO, axis=0)
    median_cr_cs_NO = np.median(sdf_change_cr_cs_NO, axis=0) 

    median_bs_no_cs = np.median(sdf_change_bs_no_cs, axis=0)
    median_cr_no_cs = np.median(sdf_change_cr_no_cs, axis=0)
    
    median_bs_no_cs_NO = np.median(sdf_change_bs_no_cs_NO, axis=0)
    median_cr_no_cs_NO = np.median(sdf_change_cr_no_cs_NO, axis=0)

    m_bs_cs, q_bs_cs = np.polyfit(trials, median_bs_cs, 1)
    m_cr_cs, q_cr_cs = np.polyfit(trials, median_cr_cs, 1)

    m_bs_cs_NO, q_bs_cs_NO = np.polyfit(trials, median_bs_cs_NO, 1)
    m_cr_cs_NO, q_cr_cs_NO = np.polyfit(trials, median_cr_cs_NO, 1)

    m_bs_no_cs, q_bs_no_cs = np.polyfit(trials, median_bs_no_cs, 1)
    m_cr_no_cs, q_cr_no_cs = np.polyfit(trials, median_cr_no_cs, 1)

    m_bs_cs_no_NO, q_bs_no_cs_NO = np.polyfit(trials, median_bs_no_cs_NO, 1)
    m_cr_cs_no_NO, q_bs_no_cs_NO = np.polyfit(trials, median_cr_no_cs_NO, 1)

     palette = list(reversed(sns.color_palette("viridis", n_trials).as_hex()))
    sm = plt.cm.ScalarMappable(
        cmap="viridis_r", norm=plt.Normalize(vmin=0, vmax=n_trials)
    )
    
    # Plot for PC CS > threshold
    plt.rcParams.update({'font.size': 16})
    fig_sdf, axs_sdf = plt.subplots(1, 2, figsize=(15, 8), sharey=True)
    
    
    for trial in range(n_trials):
        axs_sdf[0].plot(median_sdf_cs[trial], palette[trial])
        axs_sdf[1].plot(median_sdf_cs_NO[trial], palette[trial])
    
    axs_sdf[0].axvline(CS_start_first, label="CS start", linewidth=3, c=CS_color)
    axs_sdf[0].axvline(
        US_start_first - between_start, label="US start", linewidth=3, c=US_color
    )
    axs_sdf[0].axvline(
        CS_start_first + CS_burst_dur, label="CS & US end ", linewidth=3, c=CS_color
    )
    axs_sdf[1].axvline(CS_start_first, label="CS start", linewidth=3, c=CS_color)
    axs_sdf[1].axvline(
        US_start_first - between_start, label="US start", linewidth=3, c=US_color
    )
    axs_sdf[1].axvline(
        CS_start_first + CS_burst_dur, label="CS & US end ", linewidth=3, c=CS_color
    )
    
    axs_sdf[0].set_ylabel("SDF [Hz]")
    axs_sdf[0].set_xlabel("Time [ms]")
    axs_sdf[1].set_xlabel("Time [ms]")
    cbar = plt.colorbar(sm, ax=axs_sdf.ravel().tolist(), orientation='horizontal')
    cbar.set_label('Trials')
    lines1, labels1 = axs_sdf[0].get_legend_handles_labels()

    fig_sdf.legend(lines1, labels1, loc='lower center', ncol=3, 
                   #bbox_to_anchor=(0.5, 0.98), 
                   frameon=True)
    fig_sdf.suptitle(
        f"PC-CS sdf with STDP vs with NO-STDP: {noise}Hz", fontsize=18
    , fontweight='bold')
    
    plt.show()
    plt.close()
    
    # Plot for PC CS < threshold
    plt.rcParams.update({'font.size': 16})
    fig_sdf, axs_sdf = plt.subplots(1, 2, figsize=(15, 8), sharey=True)
    
    for trial in range(n_trials):
        axs_sdf[0].plot(median_sdf_no_cs[trial], palette[trial])
        axs_sdf[1].plot(median_sdf_no_cs_NO[trial], palette[trial])
    
    axs_sdf[0].axvline(CS_start_first, label="CS start", linewidth=3, c=CS_color)
    axs_sdf[0].axvline(
        US_start_first - between_start, label="US start", linewidth=3, c=US_color
    )
    axs_sdf[0].axvline(
        CS_start_first + CS_burst_dur, label="CS & US end ", linewidth=3, c=CS_color
    )
    axs_sdf[1].axvline(CS_start_first, label="CS start", linewidth=3, c=CS_color)
    axs_sdf[1].axvline(
        US_start_first - between_start, label="US start", linewidth=3, c=US_color
    )
    axs_sdf[1].axvline(
        CS_start_first + CS_burst_dur, label="CS & US end ", linewidth=3, c=CS_color
    )
    
    axs_sdf[0].set_ylabel("SDF [Hz]")
    axs_sdf[0].set_xlabel("Time [ms]")
    axs_sdf[1].set_xlabel("Time [ms]")
    #axs_sdf[0].legend()
    #axs_sdf[1].legend()
    cbar = plt.colorbar(sm, ax=axs_sdf.ravel().tolist(), orientation='horizontal')
    cbar.set_label('Trials')
    
    fig_sdf.suptitle(
        f"PC-noise sdf with STDP vs with NO-STDP: {noise}Hz", fontsize=18
    , fontweight='bold')
    lines1, labels1 = axs_sdf[0].get_legend_handles_labels()
    fig_sdf.legend(lines1, labels1, loc='lower center', ncol=3, 
                   #bbox_to_anchor=(0.5, 0.98), 
                   frameon=True)
    plt.show()
    plt.close()

    # sdf_change CS > thr
    plt.rcParams.update({'font.size': 16})
    fig, axs = plt.subplots(1, 2, sharey=True, figsize=(10, 7))
    axs[0].plot(median_bs_cs, "--o", markersize=3, color=without_NO_color, label="Baseline")
    axs[0].plot(median_cr_cs, "-o", markersize=3, color=without_NO_color, label="CR window")
    axs[0].plot(trials, trials*m_bs_cs+q_bs_cs, '-', color='grey')
    axs[0].plot(trials, trials*m_cr_cs+q_cr_cs, '-', color='grey')
    
    axs[1].plot(median_bs_cs_NO, '--o', markersize=3, color=with_NO_color, label="Baseline")
    axs[1].plot(median_cr_cs_NO, "-o", markersize=3, color=with_NO_color, label="CR window")
    axs[1].plot(trials, trials*m_bs_cs_NO+q_bs_cs_NO, '-', color='grey')
    axs[1].plot(trials, trials*m_cr_cs_NO+q_cr_cs_NO, '-', color='grey')
    
    axs[0].set_ylim(-40, 5)
    axs[0].set_xlabel("Trials")
    axs[1].set_xlabel("Trials")
    axs[0].set_ylabel("SDF change [Hz]")
    axs[0].set_title("standard STDP")
    axs[1].set_title("NO-dependent STDP")
    axs[0].legend(loc ='lower left')
    axs[1].legend(loc ='lower left')
    
    fig.suptitle(
        f"PC-CS standard STDP vs NO-dependent STDP: {noise}Hz", 
        fontsize=18, 
        fontweight='bold'
    )
    
    plt.show()
    plt.close()

    # sdf_change CS < thr
    fig, axs = plt.subplots(1, 2, sharey=True, figsize=(10, 7))
    plt.rcParams.update({'font.size': 16})
    axs[0].plot(median_bs_no_cs, "--o", markersize=3, color=without_NO_color, label="Baseline")
    axs[0].plot(median_cr_no_cs, "-o", markersize=3, color=without_NO_color, label="CR window")
    axs[0].plot(trials, trials*m_bs_no_cs+q_bs_no_cs, '-', color='grey')
    axs[0].plot(trials, trials*m_cr_no_cs+q_cr_no_cs, '-', color='grey')
    
    axs[1].plot(median_bs_no_cs_NO, '--o', markersize=3, color=with_NO_color, label="Baseline")
    axs[1].plot(median_cr_no_cs_NO, "-o", markersize=3, color=with_NO_color, label="CR window")
    axs[1].plot(trials, trials*m_bs_no_cs_NO+q_bs_no_cs_NO, '-', color='grey')
    axs[1].plot(trials, trials*m_cr_no_cs_NO+q_cr_no_cs_NO, '-', color='grey')
    
    axs[0].set_ylim(-40, 5)
    axs[0].set_xlabel("Trials")
    axs[1].set_xlabel("Trials")
    axs[0].set_ylabel("SDF change [Hz]")
    axs[0].set_title("standard STDP")
    axs[1].set_title("NO-dependent STDP")
    axs[0].legend(loc ='lower left')
    axs[1].legend(loc ='lower left')
    
    fig.suptitle(
        f"PC-noise - standard STDP vs NO-dependent STDP: {noise}Hz", 
        fontsize=16, 
        fontweight='bold'
    )
    
    plt.show()
    plt.close()

    # Boxplots for SDF changes
    plt.rcParams.update({'font.size': 16})
    fig, axs = plt.subplots(1, 2, figsize=(10, 7), sharey=True)
    positions = [1, 2, 4, 5]
    
    # CS cells boxplot

    last_trials_bs_cs = sdf_change_bs_cs[:,-5:]
    last_trials_cr_cs = sdf_change_cr_cs[:,-5:]

    last_trials_bs_cs_NO = sdf_change_bs_cs_NO[:,-5:]
    last_trials_cr_cs_NO = sdf_change_cr_cs_NO[:,-5:]
    
    boxes_cs = [last_trials_bs_cs.flatten(), last_trials_bs_cs_NO.flatten(),
                last_trials_cr_cs.flatten(), last_trials_cr_cs_NO.flatten()]
    
    print(f"Noise level (CS cells): {noise}Hz")
    stat, p = st.wilcoxon(x=last_trials_bs_cs.flatten(), y=last_trials_bs_cs_NO.flatten())
    print(f"CS baselines (without vs with NO): {p}")
    stat, p = st.wilcoxon(x=last_trials_bs_cs.flatten(), y=last_trials_cr_cs.flatten())
    print(f"CS baseline vs CR (without NO): {p}")
    stat, p = st.wilcoxon(x=last_trials_bs_cs_NO.flatten(), y=last_trials_cr_cs_NO.flatten())
    print(f"CS baseline vs CR (with NO): {p}")
    stat, p = st.wilcoxon(x=last_trials_cr_cs.flatten(), y=last_trials_cr_cs_NO.flatten())
    print(f"CS CR window (without vs with NO): {p}")
    
    bp1 = axs[0].boxplot(boxes_cs, patch_artist=True, medianprops=medianprops, positions=positions)
    for patch, color in zip(bp1['boxes'], colors*2):
        patch.set_facecolor(color)
    axs[0].axvline(3, linewidth=1, color='black')
    axs[0].set_xticks([])
    axs[0].set_ylabel("SDF change [Hz]")
    axs[0].set_ylim(-40, 5)
    axs[0].set_title(f'PC-CS - Baseline vs CR')

    # no_CS cells boxplot

    last_trials_bs_no_cs = sdf_change_bs_no_cs[:,-5:]
    last_trials_cr_no_cs = sdf_change_cr_no_cs[:,-5:]
    
    last_trials_bs_no_cs_NO = sdf_change_bs_no_cs_NO[:,-5:]
    last_trials_cr_no_cs_NO = sdf_change_cr_no_cs_NO[:,-5:]
    
    boxes_no_cs = [last_trials_bs_no_cs.flatten(), last_trials_bs_no_cs_NO.flatten(),
                   last_trials_cr_no_cs.flatten(), last_trials_cr_no_cs_NO.flatten()]
    
    print(f"Noise level (no_CS cells): {noise}Hz")
    stat, p = st.wilcoxon(x=sdf_change_bs_no_cs.flatten(), y=sdf_change_bs_no_cs_NO.flatten())
    print(f"no_CS baselines (without vs with NO): {p}")
    stat, p = st.wilcoxon(x=sdf_change_bs_no_cs.flatten(), y=sdf_change_cr_no_cs.flatten())
    print(f"no_CS baseline vs CR (without NO): {p}")
    stat, p = st.wilcoxon(x=sdf_change_bs_no_cs_NO.flatten(), y=sdf_change_cr_no_cs_NO.flatten())
    print(f"no_CS baseline vs CR (with NO): {p}")
    stat, p = st.wilcoxon(x=sdf_change_cr_no_cs.flatten(), y=sdf_change_cr_no_cs_NO.flatten())
    print(f"no_CS CR window (without vs with NO): {p}")
    
    bp2 = axs[1].boxplot(boxes_no_cs, patch_artist=True, medianprops=medianprops, positions=positions)
    for patch, color in zip(bp2['boxes'], colors*2):
        patch.set_facecolor(color)
    axs[1].axvline(3, linewidth=1, color='black')
    axs[1].set_xticks([])
    axs[1].set_ylim(-40, 5)
    axs[1].set_title(f'PC-noise - Baseline vs CR')

    fig.suptitle(
        f"noise: {noise}Hz", 
        fontsize=18, 
        fontweight='bold'
    )
    plt.tight_layout()
    plt.show()
    plt.close()
